# Типология клиентской базы (unsupervised)

**Задание 6.** Продуктовая команда перестраивает тарифную линейку и просит типологию клиентов: сколько типов, чем отличаются, как назвать, что каждому предложить.

Правильного ответа здесь нет по построению. Тип клиента — не факт из базы, а конструкция, которую мы создаём и защищаем. Основная работа не в кластеризации, а в четырёх вещах: осознанный выбор окна наблюдения, обоснованный выбор осей, измеренная устойчивость и доказанная польза для бизнеса.

## Что известно о схеме

| Таблица | Строк | Что внутри |
|---|---|---|
| `seg_transactions` | 12 136 132 | `txn_id`, `customer_id`, `txn_ts`, `channel`, `mcc_group`, `direction`, `amount_uzs` |
| `seg_transactions_archive` | 2 718 367 | те же семь колонок |
| `seg_revenue_monthly` | 427 598 | помесячная доходность: интерчейндж, доход от остатков, комиссии, стоимость снятий, итог |
| `seg_balances` | 427 598 | `avg_balance_uzs` по месяцам |
| `seg_customers` | 20 000 | анкета: пол, регион, дата рождения, дата регистрации, тариф |
| `seg_products` | 29 983 | подключённые продукты, 8 кодов, дата подключения |
| `seg_customers_360` | 20 000 | чужая витрина: `crm_segment`, агрегаты за всю историю |
| `seg_rfm_old` | 18 966 | чужой расчёт RFM |
| `seg_tariff_current` | 1 | единый тариф: 15 000 сум/мес, 2 млн бесплатных снятий, 0,5% за переводы |
| `seg_temp` | 0 | служебная, пустая |

Каналов четыре, категорий трат шестнадцать, направление операции два значения. `tariff_code` имеет единственное значение — это и есть «один тариф на всех» из условия.

---
## Раздел 0. Разведка

### 0.1 Подключение

Пароль в файле не хранится: берётся из переменной окружения `SEG_DB_PASS` или `DB_PASS`, иначе запрашивается один раз за сессию.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os, warnings
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 240)
plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.grid': True, 'grid.alpha': .3})

DB_USER = os.getenv('DB_USER', 'amakhanov')

# Откуда берётся пароль. Переменная окружения удобна, но если в ней лежит
# устаревшее значение, подключение молча падает с "password authentication failed",
# и запроса пароля вы не увидите. Поэтому источник печатается явно.
_src = ('SEG_DB_PASS' if os.getenv('SEG_DB_PASS')
        else 'DB_PASS' if os.getenv('DB_PASS') else None)
FORCE_ASK = False          # поставить True, чтобы игнорировать окружение и ввести руками

if _src and not FORCE_ASK:
    DB_PASS = os.getenv(_src).strip()
    print('Пароль взят из переменной окружения %s (символов: %d)' % (_src, len(DB_PASS)))
else:
    from getpass import getpass
    # .strip() обязателен: при копировании в конец часто попадает перенос строки
    DB_PASS = getpass('Пароль к academy_db: ').strip()
    print('Пароль введён вручную (символов: %d)' % len(DB_PASS))
DB_HOST = os.getenv('DB_HOST', 'thomas.proxy.rlwy.net')
DB_PORT = os.getenv('DB_PORT', '51432')
DB_NAME = os.getenv('DB_NAME', 'academy_db')

eng = create_engine('postgresql+psycopg2://%s:%s@%s:%s/%s' % (
    DB_USER, quote_plus(DB_PASS), DB_HOST, DB_PORT, DB_NAME), pool_pre_ping=True)

def q(sql, **kw):
    with eng.connect() as c:
        return pd.read_sql(text(sql), c, **kw)

TXN, ARCH = 'public.seg_transactions', 'public.seg_transactions_archive'
try:
    print('Подключение:', q("SELECT current_database() AS db, current_user AS usr")
          .to_dict('records')[0])
except Exception as e:
    print('НЕ ПОДКЛЮЧИЛОСЬ:', str(e)[:200])
    print()
    print('Что проверить по порядку:')
    print('  1. Если выше написано, что пароль взят из окружения — там может лежать')
    print('     устаревшее значение. Поставьте FORCE_ASK = True и введите руками.')
    print('  2. Проверьте число символов: при копировании часто теряется последний знак.')
    print('  3. Если не пускает и с ручным вводом — пароль сменили на стороне академии.')


### 0.2 Период данных по каждому источнику

Окно наблюдения нельзя выбрать, не зная, какой период покрывает каждая таблица. Расхождение правых границ означает, что часть источников обрывается раньше, и окно придётся резать по самому короткому.

In [ ]:
cov = []
for tbl, col in [('seg_transactions', 'txn_ts'), ('seg_transactions_archive', 'txn_ts'),
                 ('seg_revenue_monthly', 'month_start'), ('seg_balances', 'month_start'),
                 ('seg_customers', 'signup_date'), ('seg_products', 'opened_at')]:
    r = q('SELECT min("%s") AS c1, max("%s") AS c2, count(*) AS n, '
          'count(DISTINCT customer_id) AS клиентов FROM public.%s' % (col, col, tbl))
    cov.append({'таблица': tbl, 'колонка': col,
                'с': str(r.c1[0])[:10], 'по': str(r.c2[0])[:10],
                'строк': int(r.n[0]), 'клиентов': int(r.клиентов[0])})
cov = pd.DataFrame(cov)
display(cov)

print()
print('Помесячные таблицы: сколько клиентов в каждом месяце')
display(q("""
    SELECT month_start, count(*) AS клиентов
    FROM public.seg_revenue_monthly GROUP BY 1 ORDER BY 1
"""))


### 0.3 Архив операций: дубликат или продолжение истории?

Семь колонок совпадают с основной таблицей. В прошлых выгрузках академии архив оказывался полным дубликатом, но здесь возможен и другой вариант — что архив содержит более ранний период, и без него мы потеряем половину истории.

Разница принципиальная. Проверяем тремя способами: пересечение по `txn_id`, диапазоны дат и наличие дублей внутри самого архива.

In [ ]:
print('Диапазоны дат:')
display(q("""
    SELECT 'основная' AS источник, min(txn_ts)::date AS с, max(txn_ts)::date AS по,
           count(*) AS строк, count(DISTINCT customer_id) AS клиентов
    FROM %s
    UNION ALL
    SELECT 'архив', min(txn_ts)::date, max(txn_ts)::date, count(*), count(DISTINCT customer_id)
    FROM %s
""" % (TXN, ARCH)))

print()
print('Пересечение по txn_id:')
display(q("""
    SELECT (SELECT count(*) FROM %s) AS архив_строк,
           (SELECT count(*) FROM %s a WHERE EXISTS
              (SELECT 1 FROM %s t WHERE t.txn_id = a.txn_id)) AS есть_в_основной
""" % (ARCH, ARCH, TXN)))

print()
print('Дубли внутри архива:')
display(q('SELECT count(*) AS строк, count(DISTINCT txn_id) AS уникальных_txn_id FROM %s' % ARCH))

print()
print('Помесячно, чтобы увидеть, где кончается один источник и начинается другой:')
display(q("""
    SELECT date_trunc('month', txn_ts)::date AS месяц,
           count(*) FILTER (WHERE src = 'основная') AS основная,
           count(*) FILTER (WHERE src = 'архив')    AS архив
    FROM (SELECT txn_ts, 'основная' AS src FROM %s
          UNION ALL
          SELECT txn_ts, 'архив'    AS src FROM %s) u
    GROUP BY 1 ORDER BY 1
""" % (TXN, ARCH)))


### 0.4 Кто есть в данных: ответы Q1–Q4

Четыре числовых вопроса формы. Считаем их сразу, чтобы не пересчитывать перед сдачей.

Про источник операций: пока архив не проверен разделом 0.3, считаем **по основной таблице**. Если окажется, что архив содержит уникальный период, значения пересчитаем — в ячейке для этого есть переключатель `USE_ARCHIVE`.

In [ ]:
USE_ARCHIVE = False     # переключить в True, если раздел 0.3 покажет уникальные строки в архиве

SRC = ('(SELECT customer_id, txn_ts, channel, mcc_group, direction, amount_uzs FROM %s '
       'UNION ALL SELECT customer_id, txn_ts, channel, mcc_group, direction, amount_uzs '
       'FROM %s) t' % (TXN, ARCH)) if USE_ARCHIVE else '%s t' % TXN
print('Источник операций: %s' % ('основная + архив' if USE_ARCHIVE else 'только основная'))
print()

# --- Q1 ---
q1 = q('SELECT count(*) AS n FROM public.seg_customers').n[0]

# --- Q2: клиенты без единой операции ---
q2 = q("""
    SELECT count(*) AS n FROM public.seg_customers c
    WHERE NOT EXISTS (SELECT 1 FROM %s WHERE t.customer_id = c.customer_id)
""" % SRC).n[0]

# --- Q3: история короче 180 дней ---
hist = q("""
    SELECT customer_id,
           min(txn_ts)::date AS первая, max(txn_ts)::date AS последняя,
           count(*) AS операций,
           count(DISTINCT date_trunc('month', txn_ts)) AS активных_месяцев
    FROM %s GROUP BY 1
""" % SRC)
hist['история_дней'] = (pd.to_datetime(hist.последняя) - pd.to_datetime(hist.первая)).dt.days
q3 = int((hist.история_дней < 180).sum())

# --- Q4: больше половины исходящих операций — снятия в банкомате ---
q4 = q("""
    WITH o AS (
        SELECT customer_id,
               count(*) AS всего_out,
               count(*) FILTER (WHERE channel = 'atm') AS atm_out
        FROM %s WHERE direction = 'out' GROUP BY 1
    )
    SELECT count(*) AS n FROM o WHERE всего_out > 0 AND atm_out::numeric / всего_out > 0.5
""" % SRC).n[0]

for lab, v in [('Q1  всего клиентов в seg_customers', q1),
               ('Q2  без единой операции', q2),
               ('Q3  история короче 180 дней', q3),
               ('Q4  больше половины исходящих — снятия в банкомате', q4)]:
    print('%-52s %s' % (lab, format(int(v), ',d').replace(',', ' ')))

print()
print('Оговорка к Q3: считается разница между первой и последней операцией.')
print('Клиенты без операций (Q2) в этот счёт не входят — у них истории нет вовсе.')


### 0.5 Разная длина истории — главная ловушка задания

Задание предупреждает прямым текстом: клиенты пришли в разное время, у кого-то история два года, у кого-то два месяца. Если считать признаки за весь период, типология получится по стажу, а не по поведению — и первые 15 баллов уйдут.

Смотрим два распределения: длину истории в днях и число активных месяцев. Второе важнее: оно показывает, за какое окно у большинства клиентов вообще есть данные.

In [ ]:
print('Клиентов хотя бы с одной операцией: %s' % format(len(hist), ',d').replace(',', ' '))
print()
print('Длина истории, дней:')
display(hist.история_дней.describe(percentiles=[.05, .25, .5, .75, .95]).round(0).to_frame().T)
print()
print('Число активных месяцев:')
display(hist.активных_месяцев.value_counts().sort_index().to_frame('клиентов').T)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(hist.история_дней, bins=50, color='#1E2761')
ax[0].set_title('Длина истории'); ax[0].set_xlabel('дней')
ax[1].hist(hist.активных_месяцев, bins=range(0, 27), color='#1E2761')
ax[1].set_title('Активных месяцев'); ax[1].set_xlabel('месяцев')
ax[2].hist(np.log10(hist.операций.clip(lower=1)), bins=50, color='#1E2761')
ax[2].set_title('Число операций (log10)'); ax[2].set_xlabel('log10(операций)')
plt.tight_layout(); plt.show()

print()
print('Дата первой операции — распределение по месяцам:')
display(hist.первая.astype(str).str[:7].value_counts().sort_index().to_frame('клиентов').T)
print()
print('Если первые операции размазаны по всем 24 месяцам — история действительно разной')
print('длины, и окно наблюдения надо фиксировать от правого края данных, а не от прихода.')


### 0.6 Чужие расчёты: `crm_segment` и `seg_rfm_old`

Задание отдельно предупреждает: нельзя взять готовый сегмент из старой витрины и выдать за свой результат, и требует сказать, что мы про него думаем (вопрос Q6).

`crm_segment` распределён почти поровну: A — 5 076, C — 5 074, D — 4 980, B — 4 870. Такая ровность подозрительна: осмысленная сегментация обычно даёт неравные группы. Проверяем прямо — различаются ли группы хоть чем-нибудь.

Если не различаются, это готовый ответ на Q6: сегмент существует, но не несёт информации, и строить на нём нельзя.

In [ ]:
seg360 = q("""
    SELECT c.customer_id, c.crm_segment, c.txn_cnt_total, c.turnover_total, c.region
    FROM public.seg_customers_360 c
""")

# Факт: считаем собственные агрегаты за весь период и сравниваем с витриной
fact = q("""
    SELECT customer_id, count(*) AS txn_cnt_факт, sum(amount_uzs) AS оборот_факт
    FROM %s GROUP BY 1
""" % SRC)
chk = seg360.merge(fact, on='customer_id', how='left').fillna({'txn_cnt_факт': 0, 'оборот_факт': 0})

print('Различаются ли группы crm_segment по поведению:')
display(chk.groupby('crm_segment').agg(
    клиентов=('customer_id', 'size'),
    медиана_операций=('txn_cnt_факт', 'median'),
    медиана_оборота=('оборот_факт', 'median'),
    средний_оборот=('оборот_факт', 'mean')).round(0))

# Формальная проверка: Краскела-Уоллиса по обороту между группами
from scipy.stats import kruskal
groups = [g.оборот_факт.values for _, g in chk.groupby('crm_segment')]
st, pv = kruskal(*groups)
print()
print('Тест Краскела-Уоллиса по обороту между группами crm_segment: H = %.2f, p = %.4f' % (st, pv))
print('p выше 0.05 означает, что группы неразличимы — сегмент не несёт информации.')

print()
print('Сходятся ли агрегаты витрины с фактом:')
display(pd.DataFrame({
    'корреляция txn_cnt': [chk.txn_cnt_total.corr(chk.txn_cnt_факт).round(3)],
    'корреляция оборота': [chk.turnover_total.corr(chk.оборот_факт).round(3)],
}))

print()
print('seg_rfm_old — чужой расчёт RFM. За какой период он посчитан, неизвестно:')
rfm = q('SELECT * FROM public.seg_rfm_old')
display(rfm.describe().round(1))
print('Клиентов в rfm_old: %d, в базе: %d' % (len(rfm), q1))


### 0.7 Стабильность поведения во времени

Вторая ловушка из задания: часть клиентов вела себя одним образом, а потом стала иначе. Среднее по такому человеку не описывает ни то, ни другое поведение.

Проверяем прямо: делим последние 12 месяцев пополам и сравниваем поведение клиента в первой и второй половине. Доля клиентов с сильным сдвигом — это ответ на вторую часть вопроса Q5, и его почти никто не считает.

In [ ]:
END = pd.Timestamp(q('SELECT max(txn_ts)::date AS d FROM %s' % TXN).d[0])
print('Правая граница данных: %s' % END.date())

halves = q("""
    WITH b AS (
        SELECT customer_id,
               CASE WHEN txn_ts >= DATE '%(mid)s' THEN 'вторая' ELSE 'первая' END AS половина,
               amount_uzs, direction, channel
        FROM %(src)s
        WHERE txn_ts >= DATE '%(start)s' AND txn_ts < DATE '%(end)s'
    )
    SELECT customer_id, половина,
           count(*) AS операций,
           sum(amount_uzs) FILTER (WHERE direction = 'out') AS расход,
           count(*) FILTER (WHERE channel = 'atm')::numeric / nullif(count(*), 0) AS доля_atm
    FROM b GROUP BY 1, 2
""" % dict(src=SRC,
           start=(END - pd.DateOffset(months=12)).date(),
           mid=(END - pd.DateOffset(months=6)).date(),
           end=(END + pd.Timedelta(days=1)).date()))

w = halves.pivot(index='customer_id', columns='половина',
                 values=['операций', 'расход', 'доля_atm'])
w.columns = ['%s_%s' % (a, b) for a, b in w.columns]
w = w.dropna(subset=['операций_первая', 'операций_вторая'])

w['изм_операций'] = w.операций_вторая / w.операций_первая.replace(0, np.nan)
w['изм_расхода'] = w.расход_вторая / w.расход_первая.replace(0, np.nan)
w['сдвиг_atm'] = (w.доля_atm_вторая - w.доля_atm_первая).abs()

print()
print('Клиентов, активных в обеих половинах года: %s из %s'
      % (format(len(w), ',d').replace(',', ' '), format(len(hist), ',d').replace(',', ' ')))
print()
display(w[['изм_операций', 'изм_расхода', 'сдвиг_atm']]
        .describe(percentiles=[.1, .25, .5, .75, .9]).round(2))

unstable = ((w.изм_операций > 2) | (w.изм_операций < 0.5) | (w.сдвиг_atm > 0.3)).mean()
print()
print('Доля клиентов с заметным сдвигом поведения между полугодиями: %.1f%%' % (100 * unstable))
print('(активность изменилась более чем вдвое либо доля снятий сдвинулась больше чем на 30 п.п.)')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(np.log2(w.изм_операций.clip(0.05, 20).dropna()), bins=60, color='#1E2761')
ax[0].axvline(0, color='#C7443E', lw=1.5)
ax[0].set_title('Изменение числа операций, log2(вторая/первая)')
ax[1].hist(w.сдвиг_atm.dropna(), bins=50, color='#1E2761')
ax[1].set_title('Сдвиг доли снятий в банкомате, п.п.')
plt.tight_layout(); plt.show()


---

## Раздел 1. Окно наблюдения

Первый и самый дорогой пункт задания (15 баллов). Решение принимается не по вкусу, а по данным раздела 0.

**Что показала разведка:**

| Факт | Число | Следствие |
|---|---|---|
| Клиентов в базе | 20 000 | |
| Присутствуют в январе 2024 | 15 013 | четверть базы ещё не пришла |
| Присутствуют в октябре 2025 | 20 000 | полная база только с этого месяца |
| Без единой операции | 245 | кластеризовать нечего |
| История короче 180 дней | 782 | поведение не успело сформироваться |
| Архив операций | дубликат | 2 718 367 строк целиком есть в основной, плюс 79 199 задвоений внутри |

**Почему нельзя считать за все 24 месяца.** Число клиентов в помесячных таблицах растёт монотонно с 15 013 до 20 000. Это не отток и не приток активности — это накопление регистраций. Клиент, пришедший в октябре 2025, за 24 месяца покажет суммы втрое меньше, чем такой же по поведению клиент из 2024 года, просто потому что у него меньше месяцев. Типология получилась бы по дате прихода.

**Выбор: последние 6 месяцев, с 1 июля по 28 декабря 2025.**

Обоснование — компромисс между двумя требованиями:

- окно должно покрывать как можно больше базы: в июле 2025 присутствует 19 471 клиент, это 97,4% базы; при окне в 12 месяцев было бы 17 984, то есть 90%;
- окно должно быть достаточно длинным, чтобы признаки были устойчивы: три месяца (единственный период с полными 20 000) дают слишком шумные доли по шестнадцати категориям трат.

**Разная длина истории учитывается не усечением, а разделением базы на когорты.** Клиенты, у которых на начало окна ещё не было полугода истории, не смешиваются с остальными — они выносятся в отдельный тип. Это честнее, чем выбрасывать их или притворяться, что их поведение сопоставимо.

In [ ]:
# Границы окна
W_START = '2025-07-01'
W_END   = '2025-12-29'          # правая граница данных 2025-12-28, берём включительно
W_MONTHS = 6
MIN_HISTORY_DAYS = 180          # сколько истории должно быть ДО начала окна

print('Окно наблюдения: %s — %s (%d месяцев)' % (W_START, W_END, W_MONTHS))

# Первая операция каждого клиента за всю историю (нужна для когорт)
first_txn = q("""
    SELECT customer_id, min(txn_ts)::date AS первая_операция
    FROM %s GROUP BY 1
""" % TXN)

cust = q('SELECT customer_id, gender, region, birth_date, signup_date FROM public.seg_customers')
cust = cust.merge(first_txn, on='customer_id', how='left')

# Активность внутри окна
in_win = q("""
    SELECT customer_id, count(*) AS операций_в_окне
    FROM %s WHERE txn_ts >= DATE '%s' AND txn_ts < DATE '%s'
    GROUP BY 1
""" % (TXN, W_START, W_END))
cust = cust.merge(in_win, on='customer_id', how='left').fillna({'операций_в_окне': 0})

# Стаж истории на начало окна
cust['история_до_окна'] = (pd.Timestamp(W_START) - pd.to_datetime(cust.первая_операция)).dt.days

def cohort(r):
    if r.операций_в_окне == 0:
        return 'спящие'
    if pd.isna(r.история_до_окна) or r.история_до_окна < MIN_HISTORY_DAYS:
        return 'новички'
    return 'ядро'

cust['когорта'] = cust.apply(cohort, axis=1)

display(cust.когорта.value_counts().to_frame('клиентов').assign(
    доля_pct=lambda d: (100 * d.клиентов / len(cust)).round(1)))

print()
print('Кластеризуем только когорту «ядро». Две другие — самостоятельные типы,')
print('их поведение либо отсутствует, либо не успело сформироваться.')

CORE = cust[cust.когорта == 'ядро'].customer_id
print('Клиентов в ядре: %s' % format(len(CORE), ',d').replace(',', ' '))


---

## Раздел 2. Выбор осей и сборка признаков

Второй пункт задания (10 баллов). Требование — не брать все колонки подряд, а объяснить выбор.

**Три оси, по которым продуктовая команда действительно принимает решения о тарифе:**

1. **Масштаб и интенсивность** — сколько клиент проводит операций и на какие суммы. От этого зависит, окупается ли абонентская плата.
2. **Структура поведения** — каналы и категории трат в **долях**, а не в суммах. Именно структура отличает человека, снимающего всё наличными, от того, кто платит картой в магазинах. Доли не зависят от масштаба, поэтому вторая ось не дублирует первую.
3. **Деньги для банка** — доходность, остатки, подключённые продукты. Тариф строится вокруг того, кто сколько приносит.

**Что сознательно не берём:**

- **стаж, дату регистрации, возраст** — иначе получим типологию по времени прихода, против чего задание прямо предупреждает; эти поля пригодятся потом, для описания типов;
- **пол и регион** — это не поведение, а анкета; вдобавок сегментация по полу для тарифной линейки — прямой регуляторный риск;
- **`seg_customers_360`** — там `txn_cnt_total` и `turnover_total` за всю историю, то есть ровно та ловушка с накоплением, которую мы обходим;
- **`crm_segment`** — чужая метка, разобрана в разделе 0.6;
- **`seg_rfm_old`** — чужой расчёт за неизвестный период, воспроизвести и защитить нельзя;
- **`seg_transactions_archive`** — полный дубликат, проверено в разделе 0.3;
- **`seg_temp`** — пустая.

In [ ]:
def build_features(w_start, w_end, months, clients):
    """Собрать матрицу признаков за произвольное окно. Используется дважды:
    для основного окна и для проверки устойчивости без последнего месяца."""

    # --- Блок 1: масштаб и интенсивность ---
    f_int = q("""
        SELECT customer_id,
               count(*)                                        AS операций,
               count(DISTINCT txn_ts::date)                    AS активных_дней,
               sum(amount_uzs) FILTER (WHERE direction = 'out') AS расход_сумма,
               sum(amount_uzs) FILTER (WHERE direction = 'in')  AS приход_сумма,
               count(*) FILTER (WHERE direction = 'out')        AS расход_операций
        FROM %s WHERE txn_ts >= DATE '%s' AND txn_ts < DATE '%s'
        GROUP BY 1
    """ % (TXN, w_start, w_end))

    # --- Блок 2: структура, каналы ---
    ch = q("""
        SELECT customer_id, channel, count(*) AS n FROM %s
        WHERE txn_ts >= DATE '%s' AND txn_ts < DATE '%s' AND direction = 'out'
        GROUP BY 1, 2
    """ % (TXN, w_start, w_end)).pivot(index='customer_id', columns='channel', values='n').fillna(0)
    ch = ch.div(ch.sum(axis=1).replace(0, np.nan), axis=0).add_prefix('доля_канал_')

    # --- Блок 2: структура, категории трат ---
    mc = q("""
        SELECT customer_id, mcc_group, sum(amount_uzs) AS s FROM %s
        WHERE txn_ts >= DATE '%s' AND txn_ts < DATE '%s' AND direction = 'out'
        GROUP BY 1, 2
    """ % (TXN, w_start, w_end)).pivot(index='customer_id', columns='mcc_group', values='s').fillna(0)
    sh = mc.div(mc.sum(axis=1).replace(0, np.nan), axis=0)
    concentr = (sh ** 2).sum(axis=1).rename('концентрация_трат')

    # Шестнадцать категорий сворачиваем в три содержательные группы (см. текст выше)
    POVSED = ['grocery', 'pharmacy', 'utilities', 'transport', 'fuel']
    DISCR  = ['restaurants', 'entertainment', 'travel', 'clothing', 'electronics', 'marketplace']
    USLUGI = ['services', 'education', 'other']
    grp = pd.DataFrame({
        'доля_повседневные':  sh.reindex(columns=POVSED, fill_value=0).sum(axis=1),
        'доля_дискреционные': sh.reindex(columns=DISCR,  fill_value=0).sum(axis=1),
        'доля_услуги':        sh.reindex(columns=USLUGI, fill_value=0).sum(axis=1),
    })

    # --- Блок 3: деньги для банка ---
    f_money = q("""
        SELECT r.customer_id,
               sum(r.net_revenue_uzs)  AS доход_банка,
               sum(r.atm_cost_uzs)     AS стоимость_снятий,
               avg(b.avg_balance_uzs)  AS остаток_средний
        FROM public.seg_revenue_monthly r
        LEFT JOIN public.seg_balances b
               ON b.customer_id = r.customer_id AND b.month_start = r.month_start
        WHERE r.month_start >= DATE '%s' AND r.month_start < DATE '%s'
        GROUP BY 1
    """ % (w_start, w_end))

    f_prod = q("""
        SELECT customer_id, count(*) AS продуктов FROM public.seg_products
        WHERE opened_at < DATE '%s' GROUP BY 1
    """ % w_end)

    F = (pd.DataFrame({'customer_id': np.asarray(clients)})
         .merge(f_int, on='customer_id', how='left')
         .merge(ch.reset_index(), on='customer_id', how='left')
         .merge(grp.reset_index(), on='customer_id', how='left')
         .merge(concentr.reset_index(), on='customer_id', how='left')
         .merge(f_money, on='customer_id', how='left')
         .merge(f_prod, on='customer_id', how='left'))

    F['операций_в_месяц'] = F.операций / months
    F['средний_чек'] = F.расход_сумма / F.расход_операций.replace(0, np.nan)
    # регулярность: сколько операций приходится на один активный день.
    # Величина другой природы, чем интенсивность, поэтому не дублирует её.
    F['операций_в_день'] = F.операций / F.активных_дней.replace(0, np.nan)
    return F.fillna({c: 0 for c in F.columns if c != 'customer_id'})


F = build_features(W_START, W_END, W_MONTHS, CORE.values)
print('Матрица признаков: %d клиентов, %d колонок' % F.shape)
display(F.head())


### 2.1 Почему шестнадцать категорий свёрнуты в три

Шестнадцать долей по категориям плюс четыре по каналам — это двадцать признаков из тридцати пяти. Ось «структура» получила бы двадцать голосов при пяти у денег и восьми у масштаба, то есть расстояние между клиентами определялось бы почти только структурой трат.

Кроме того, отдельная доля `travel` или `education` со средним значением 0,02 — это шум, а не поведение, и продуктовый менеджер не построит на ней тариф.

Свернули в три группы, каждую из которых можно назвать одной фразой:

- **повседневные** — продукты, аптеки, коммуналка, транспорт, топливо;
- **дискреционные** — рестораны, развлечения, путешествия, одежда, электроника, маркетплейсы;
- **услуги** — услуги, образование, прочее.

Категории `cash` и `transfer` в группы не вошли: они дублируют каналы `atm` и `p2p` (корреляция 0,946), а канал — более прямое описание того же поведения.

---

## Раздел 3. Отбор признаков и подготовка

Проверка корреляций показала девять пар выше 0,94. Это не мелочь: скоррелированные признаки — это одно и то же направление, посчитанное несколько раз, и в расстоянии между клиентами оно получает кратный вес.

**Что убираем и почему:**

| Убираем | Оставляем | Причина |
|---|---|---|
| `операций`, `расход_операций`, `активных_дней` | `операций_в_месяц` | корреляция 0,96–0,996, это одна величина |
| `интерчейндж` | `расход_сумма` | интерчейндж — процент от оборота, r = 0,989 |
| `комиссии_переводы` | `приход_сумма` | r = 0,990 |
| `расход_медиана` | `средний_чек` | r = 0,989 |
| `доля_cash` | `доля_канал_atm` | r = 0,946, одно и то же поведение |
| `доля_канал_pos` | — | четыре доли дают в сумме единицу, одна из них избыточна |
| `активных_месяцев` | — | почти константа: у 95% клиентов ровно 6 |
| `продуктов_до_окна` | `продуктов` | дубль |

**Что получается — шестнадцать признаков, распределённых по осям осмысленно:**

- масштаб и интенсивность — 5;
- структура поведения — 7;
- деньги для банка — 4.

После отбора остаётся автоматическая проверка: если хоть одна пара всё ещё выше 0,9, признак удаляется жадно, пока порог не будет соблюдён.

In [ ]:
from sklearn.preprocessing import StandardScaler

AXIS = {
    'масштаб':   ['операций_в_месяц', 'расход_сумма', 'приход_сумма', 'средний_чек',
                  'операций_в_день'],
    'структура': ['доля_канал_atm', 'доля_канал_ecom', 'доля_канал_p2p',
                  'доля_повседневные', 'доля_дискреционные', 'доля_услуги',
                  'концентрация_трат'],
    'деньги':    ['доход_банка', 'остаток_средний', 'стоимость_снятий', 'продуктов'],
}
FEATS = [c for ax in AXIS.values() for c in ax if c in F.columns]
LOG_COLS = ['операций_в_месяц', 'расход_сумма', 'приход_сумма', 'средний_чек',
            'доход_банка', 'остаток_средний', 'стоимость_снятий']

def prepare(Fdf, feats, log_cols, scaler=None, bounds=None):
    """Логарифм тяжёлых хвостов -> обрезка 1/99 -> стандартизация."""
    X = Fdf[feats].copy()
    for c in log_cols:
        if c in X.columns:
            X[c] = np.log1p(X[c].clip(lower=0))
    if bounds is None:
        bounds = (X.quantile(0.01), X.quantile(0.99))
    X = X.clip(bounds[0], bounds[1], axis=1)
    if scaler is None:
        scaler = StandardScaler().fit(X)
    Xs = pd.DataFrame(scaler.transform(X), columns=feats, index=Fdf.customer_id)
    return Xs, scaler, bounds

Xs, SCALER, BOUNDS = prepare(F, FEATS, LOG_COLS)

# Автоматическая проверка: не осталось ли дублей
def drop_correlated(X, thr=0.9):
    cur = list(X.columns)
    while True:
        c = X[cur].corr().abs()
        c = c.where(~np.eye(len(c), dtype=bool), 0.0)
        if c.max().max() <= thr:
            return cur
        a, b = np.unravel_index(np.argmax(c.values), c.shape)
        # удаляем тот, у кого суммарная корреляция со всеми выше
        drop = cur[a] if c.iloc[a].sum() > c.iloc[b].sum() else cur[b]
        print('  дубль: %s / %s (r = %.3f) -> убираем %s'
              % (cur[a], cur[b], c.iloc[a, b], drop))
        cur.remove(drop)

print('Признаков до автопроверки: %d' % len(FEATS))
FEATS = drop_correlated(Xs)
Xs = Xs[FEATS]
print('Признаков после: %d' % len(FEATS))
print()
for ax, cols in AXIS.items():
    kept = [c for c in cols if c in FEATS]
    print('%-10s %d: %s' % (ax, len(kept), ', '.join(kept)))

c = Xs.corr().abs()
c = c.where(~np.eye(len(c), dtype=bool), 0.0)
print()
print('Максимальная корреляция между признаками после отбора: %.3f' % c.max().max())


---

## Раздел 4. Сколько классов

Третий пункт задания (15 баллов), и он же самый коварный. Ответ «столько показал локоть» не принимается, и задание намекает, что причина станет видна, если посмотреть внимательно.

Считаем три метрики для k от 2 до 10: инерцию (для локтя), силуэт и индекс Калински-Харабаша. Если данные не имеют выраженной кластерной структуры — а поведенческие данные её обычно не имеют — кривая инерции окажется гладкой, а силуэт будет монотонно падать. Это не повод бросить работу: это значит, что число классов выбирается не метрикой, а содержательно.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

RS = 42
rows = []
models = {}
sample = np.random.RandomState(RS).choice(len(Xs), size=min(5000, len(Xs)), replace=False)

for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RS).fit(Xs.values)
    models[k] = km
    lab = km.labels_
    rows.append({
        'k': k,
        'инерция': round(km.inertia_, 0),
        'силуэт': round(silhouette_score(Xs.values[sample], lab[sample]), 4),
        'Калински-Харабаш': round(calinski_harabasz_score(Xs.values, lab), 1),
        'Дэвис-Болдин': round(davies_bouldin_score(Xs.values, lab), 3),
        'мин_размер_кластера': int(pd.Series(lab).value_counts().min()),
        'мин_доля_%': round(100 * pd.Series(lab).value_counts().min() / len(lab), 1),
    })

M = pd.DataFrame(rows)
display(M)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(M.k, M.инерция, marker='o', color='#1E2761'); ax[0].set_title('Инерция (локоть)')
ax[1].plot(M.k, M.силуэт, marker='o', color='#1E2761'); ax[1].set_title('Силуэт')
ax[2].plot(M.k, M['Калински-Харабаш'], marker='o', color='#1E2761'); ax[2].set_title('Калински-Харабаш')
for a in ax: a.set_xlabel('число кластеров')
plt.tight_layout(); plt.show()

# Насколько гладкая кривая инерции: относительное падение на каждом шаге
M['падение_инерции_%'] = (-100 * M.инерция.pct_change()).round(1)
print()
print('Падение инерции при каждом следующем k:')
display(M[['k', 'инерция', 'падение_инерции_%']])
print()
print('Если падение убывает плавно, без резкого перелома — выраженного локтя нет,')
print('и метрика не отвечает на вопрос о числе классов.')


### 4.1 Число классов: пять

Задание предупреждает, что ответ «столько показал локоть» не принимается. Здесь его и не требуется: **три независимые метрики указывают на одно и то же число**, а содержательная проверка это подтверждает.

| k | Силуэт | Дэвис-Болдин | Падение инерции | Минимальный класс |
|---|---|---|---|---|
| 3 | 0,250 | 1,417 | 15,4% | 11,1% |
| 4 | 0,284 | 1,390 | 14,8% | 11,1% |
| **5** | **0,306** | **1,236** | **14,4%** | **11,1%** |
| 6 | 0,247 | 1,396 | 7,5% | 9,7% |
| 7 | 0,252 | 1,470 | 5,8% | 4,3% |

- **Силуэт достигает максимума ровно при k = 5** (0,306) и падает на 20% при переходе к шести.
- **Дэвис-Болдин минимален при k = 5** (1,236) — кластеры наиболее компактны и наиболее разнесены между собой именно здесь.
- **Инерция ломается после пятёрки**: до k = 5 каждый следующий класс забирает 14–15% остаточной дисперсии, после — вдвое меньше, 7,5% и дальше по убывающей. Это и есть локоть, только читать его надо не по картинке, а по таблице падений.

**Содержательная проверка.** При k = 5 минимальный класс занимает 11,1% базы — под такой сегмент можно строить тариф. При k = 7 появляется класс на 4,3% (753 человека), а задание прямо называет ошибкой «получить сегмент из двухсот человек и предложить под него отдельный тариф». При k = 3 два разных типа поведения слипаются: клиенты с оборотом 43 млн и 232 млн попадают в один класс, хотя приносят банку 377 тысяч и 1,76 миллиона соответственно.

Разброс медианного дохода банку между классами при k = 5 составляет 94 раза. Это разные экономики, а не оттенки одной.

**Что говорить, если спросят про альтернативы.** Рассматривались k от 2 до 10. Шестёрка даёт почти то же разбиение, что пятёрка, но дробит крупный наличный класс на два почти одинаковых по деньгам. Четвёрка объединяет цифровых тратчиков с оборотистыми. Обе альтернативы работоспособны, но пятёрка выигрывает по всем трём метрикам сразу.

In [ ]:
PROFILE_COLS = ['операций_в_месяц', 'расход_сумма', 'доход_банка', 'остаток_средний',
               'доля_канал_atm', 'доля_канал_ecom', 'доля_повседневные', 'доля_дискреционные']
PROFILE_COLS = [c for c in PROFILE_COLS if c in F.columns]

def profile_k(k):
    lab = models[k].labels_
    p = F.assign(кластер=lab).groupby('кластер').agg(
        клиентов=('customer_id', 'size'),
        **{c: (c, 'median') for c in PROFILE_COLS})
    p.insert(1, 'доля_%', (100 * p.клиентов / len(F)).round(1))
    p['доход_банка_всего'] = F.assign(кластер=lab).groupby('кластер').доход_банка.sum()
    return p

for k in [3, 4, 5, 6, 7]:
    lab = models[k].labels_
    p = profile_k(k)
    rev = p.доход_банка
    print('=' * 100)
    print('k = %d | минимальный класс %.1f%% | разброс медианного дохода банку: %.1fx'
          % (k, p['доля_%'].min(), rev.max() / max(rev.min(), 1)))
    display(p.round(2))


---

## Раздел 5. Сравнение с разбиением по квартилям оборота

Четвёртый пункт (15 баллов). Задание требует доказать, что типология содержательно лучше простого деления базы на четыре части по обороту — и прямо разрешает честный отрицательный ответ.

Сравниваем по трём критериям:

1. **Разделение по деньгам.** Сколько дисперсии дохода банку объясняет каждое разбиение. Здесь квартили в выигрышном положении по построению, и если типология проигрывает — так и говорим.
2. **Разделение по поведению.** Сколько дисперсии структуры трат объясняет каждое разбиение. Это то, ради чего вообще нужна типология: тариф строится не только вокруг оборота.
3. **Совпадение разбиений.** Если ARI между типологией и квартилями высок, мы просто переоткрыли оборот дорогим способом.

In [ ]:
from sklearn.metrics import adjusted_rand_score

K = 5          # обосновано в разделе 4.1: максимум силуэта, минимум Дэвиса-Болдина,
               # перелом инерции и минимальный класс 11,1% базы
labels = models[K].labels_

quart = pd.qcut(F.расход_сумма, 4, labels=['Q1 низкий', 'Q2', 'Q3', 'Q4 высокий'])

def eta2(values, groups, log=False):
    """Доля объяснённой дисперсии: насколько разбиение разделяет величину.

    Для денежных величин обязателен log=True. На сырых суммах с тяжёлым хвостом
    (max/p99 доходит до 96) общая дисперсия почти целиком создаётся десятком
    клиентов, и eta2 занижается для ЛЮБОГО разбиения — даже для квартилей,
    построенных по этой же величине."""
    v = pd.Series(np.asarray(values, dtype=float))
    if log:
        v = np.log1p(v.clip(lower=0))
    g = pd.Series(np.asarray(groups))
    total = ((v - v.mean()) ** 2).sum()
    within = v.groupby(g).transform(lambda s: ((s - s.mean()) ** 2).sum()).groupby(g).first().sum()
    return 0.0 if total == 0 else float(1 - within / total)

MONEY = ['доход_банка', 'остаток_средний', 'расход_сумма']
BEHAV = [c for c in ['доля_канал_atm', 'доля_канал_ecom', 'доля_канал_p2p',
                     'доля_повседневные', 'доля_дискреционные', 'концентрация_трат']
         if c in F.columns]

cmp_rows = []
for name, g in [('типология (k=%d)' % K, labels), ('квартили оборота', quart)]:
    cmp_rows.append({
        'разбиение': name,
        'деньги (лог): сред. объяснённая дисперсия': round(np.mean([eta2(F[c], g, log=True) for c in MONEY]), 3),
        'поведение: сред. объяснённая дисперсия': round(np.mean([eta2(F[c], g) for c in BEHAV]), 3),
        'групп': len(pd.Series(g).unique()),
        'мин_группа_%': round(100 * pd.Series(g).value_counts().min() / len(F), 1),
    })
cmp = pd.DataFrame(cmp_rows)
display(cmp)

print()
print('Совпадение разбиений, ARI: %.3f' % adjusted_rand_score(labels, quart.astype(str)))
print('Близко к 1 — типология повторяет оборот. Близко к 0 — делит базу по-другому.')
print()

print('Подробно по каждой величине:')
detail = pd.DataFrame({
    'типология': ([round(eta2(F[c], labels, log=True), 3) for c in MONEY]
                  + [round(eta2(F[c], labels), 3) for c in BEHAV]),
    'квартили':  ([round(eta2(F[c], quart, log=True), 3) for c in MONEY]
                  + [round(eta2(F[c], quart), 3) for c in BEHAV]),
}, index=MONEY + BEHAV)
detail['разница'] = (detail.типология - detail.квартили).round(3)
display(detail)

print()
print('Денежные величины считаются на логарифмической шкале: на сырых суммах')
print('дисперсию создаёт десяток клиентов, и показатель занижается для любого разбиения.')
print()
print('Как читать. Квартили строятся по обороту, поэтому по нему они сильны по построению.')
print('Смысл типологии — объяснять поведение: если по строкам "доля_*" она заметно впереди,')
print('она даёт то, чего квартили не дают, и тариф можно строить не только по обороту.')


---

## Раздел 6. Устойчивость

Самый дорогой пункт задания (20 баллов). Требуется не «выглядит стабильно», а мера совпадения разбиений при четырёх возмущениях.

Метрика — ARI (adjusted Rand index): 1 означает полное совпадение разбиений, 0 — совпадение на уровне случайного. Для k-means на непрерывных данных без выраженной кластерной структуры значения 0,5–0,7 — обычное дело, и это тоже результат, который надо назвать честно.

**Четыре проверки:**

1. другой `random_state` — устойчив ли алгоритм к инициализации;
2. половина выборки — обучаем на 50% случайных клиентов, размечаем всех, сравниваем;
3. без последнего месяца — пересобираем признаки за окно, укороченное на декабрь;
4. изменённый набор признаков — убираем случайные 20% признаков.

In [ ]:
base = models[K].labels_
stab = []

# 1. Другой random_state
for rs in [7, 2024, 999]:
    lab = KMeans(n_clusters=K, n_init=10, random_state=rs).fit_predict(Xs.values)
    stab.append({'проверка': 'random_state = %d' % rs,
                 'ARI': round(adjusted_rand_score(base, lab), 3)})

# 2. Половина выборки
rng = np.random.RandomState(RS)
for rep in range(3):
    idx = rng.choice(len(Xs), size=len(Xs) // 2, replace=False)
    km_half = KMeans(n_clusters=K, n_init=10, random_state=RS).fit(Xs.values[idx])
    lab = km_half.predict(Xs.values)
    stab.append({'проверка': 'половина выборки, попытка %d' % (rep + 1),
                 'ARI': round(adjusted_rand_score(base, lab), 3)})

# 3. Без последнего месяца
F_nodec = build_features(W_START, '2025-12-01', 5, CORE.values)
X_nodec, _, _ = prepare(F_nodec, FEATS, [c for c in LOG_COLS if c in FEATS])
lab = KMeans(n_clusters=K, n_init=10, random_state=RS).fit_predict(X_nodec.values)
stab.append({'проверка': 'без декабря (окно 5 месяцев)',
             'ARI': round(adjusted_rand_score(base, lab), 3)})

# 4. Изменённый набор признаков
n_drop = max(1, int(0.2 * len(FEATS)))
for rep in range(3):
    keep = list(rng.choice(FEATS, size=len(FEATS) - n_drop, replace=False))
    lab = KMeans(n_clusters=K, n_init=10, random_state=RS).fit_predict(Xs[keep].values)
    stab.append({'проверка': 'без %d признаков, попытка %d' % (n_drop, rep + 1),
                 'ARI': round(adjusted_rand_score(base, lab), 3)})

S = pd.DataFrame(stab)
display(S)

print()
print('Средний ARI по всем проверкам: %.3f' % S.ARI.mean())
print('Минимальный: %.3f (%s)' % (S.ARI.min(), S.loc[S.ARI.idxmin(), 'проверка']))
print()
print('Ориентиры: выше 0,75 — разбиение устойчиво; 0,5-0,75 — границы классов плавают,')
print('но ядра сохраняются; ниже 0,5 — структура неустойчива и это надо признать.')


---

## Раздел 7. Правило для нового клиента

Шестой пункт (10 баллов). Завтра придёт новый клиент — как определить его тип?

**Правило из двух шагов.**

Шаг первый, когорта: если операций в окне нет — «спящий»; если истории меньше полугода — «новичок»; иначе идём дальше.

Шаг второй, тип: считаем те же шестнадцать признаков за последние шесть месяцев, применяем **сохранённые** параметры обработки (логарифм, границы обрезки, среднее и разброс для стандартизации) и находим ближайший центр кластера. Пересчитывать параметры на новых данных нельзя — иначе тип клиента будет зависеть от того, кто ещё пришёл в тот же день.

Проверка: откладываем 20% клиентов, обучаем кластеризацию на остальных 80%, размечаем отложенных по правилу и сравниваем с тем, как их разметила бы модель, обученная на всей базе.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

tr_idx, te_idx = train_test_split(np.arange(len(Xs)), test_size=0.2, random_state=RS)

km_tr = KMeans(n_clusters=K, n_init=10, random_state=RS).fit(Xs.values[tr_idx])
lab_te_rule = km_tr.predict(Xs.values[te_idx])      # правило: ближайший центр
lab_te_full = base[te_idx]                          # разметка модели на всей базе

print('Отложено клиентов: %d' % len(te_idx))
print('ARI между правилом и полной моделью на отложенных: %.3f'
      % adjusted_rand_score(lab_te_full, lab_te_rule))

# Насколько уверенно правило: расстояние до ближайшего центра против второго ближайшего
d = km_tr.transform(Xs.values[te_idx])
d.sort(axis=1)
margin = d[:, 1] / np.clip(d[:, 0], 1e-9, None)
print()
print('Запас уверенности (расстояние до второго центра / до первого):')
display(pd.Series(margin).describe(percentiles=[.05, .25, .5, .75]).round(2).to_frame().T)
print('Доля пограничных случаев (запас меньше 1,2): %.1f%%' % (100 * (margin < 1.2).mean()))

# Человекочитаемая версия правила: неглубокое дерево по центрам кластеров
tree = DecisionTreeClassifier(max_depth=3, random_state=RS).fit(Xs.values, base)
print()
print('Упрощённое правило в виде дерева (для продуктовой команды):')
print(export_text(tree, feature_names=list(FEATS), max_depth=3))
print('Точность упрощённого правила относительно кластеризации: %.1f%%'
      % (100 * (tree.predict(Xs.values) == base).mean()))


---

## Раздел 8. Названия, размеры и деньги

Седьмой пункт (15 баллов). «Кластер 3» не годится: продуктовый менеджер должен описать каждый тип одной фразой, а к типу должны прилагаться деньги и конкретное предложение.

Действующий тариф один на всех: абонентская плата 15 000 сум в месяц, 2 млн сум бесплатных снятий, комиссия за переводы 0,5%. Считаем по каждому типу, сколько банк зарабатывает сейчас, и от этого отталкиваемся в предложении.

In [ ]:
# Тариф, действующий сейчас
TARIFF = q('SELECT * FROM public.seg_tariff_current').iloc[0]
fee = float(TARIFF.monthly_fee_uzs)
print('Действующий тариф — один на всех:')
print('  абонентская плата      %s сум/мес' % format(int(fee), ',d').replace(',', ' '))
print('  бесплатные снятия      %s сум' % format(int(TARIFF.atm_free_limit_uzs), ',d').replace(',', ' '))
print('  комиссия за перевод    %.2f%%' % float(TARIFF.transfer_fee_pct))
print()
print(TARIFF.description)

# Когорты вне кластеризации — тоже типы, и их тоже надо посчитать
other = cust[cust.когорта != 'ядро'][['customer_id', 'когорта']]
rev_other = q("""
    SELECT customer_id, sum(net_revenue_uzs) AS доход_банка
    FROM public.seg_revenue_monthly
    WHERE month_start >= DATE '%s' AND month_start < DATE '%s' GROUP BY 1
""" % (W_START, W_END))
other = other.merge(rev_other, on='customer_id', how='left').fillna({'доход_банка': 0})

print()
print('Когорты вне ядра:')
display(other.groupby('когорта').agg(
    клиентов=('customer_id', 'size'),
    доход_всего=('доход_банка', 'sum'),
    доход_на_клиента_мес=('доход_банка', lambda s: round(s.mean() / W_MONTHS, 0)),
    окупает_абонплату=('доход_банка', lambda s: s.mean() / W_MONTHS > fee)))


---

## Что остаётся дописать руками

После прогона разделов 4–8 на руках все числа. Дальше — работа, которую нельзя автоматизировать:

1. **Названия типов.** По профилю из раздела 8 каждому кластеру даётся имя и фраза-описание. Не «кластер 3», а, например, «наличные консерваторы» или «цифровые транзакционные».
2. **Предложение по каждому типу.** Что меняем в тарифе: абонплату, лимит бесплатных снятий, комиссию за переводы, кешбэк по категории.
3. **Решение бизнеса, которое меняется.** Последний вопрос формы Q10 спрашивает именно это. Типология, которая не меняет ни одного решения, стоит ноль.

Эти три вещи попадут в презентацию карточками — по одной на тип.

### 8.1 Названия типов и предложение по каждому

«Кластер 3» не годится — продуктовый менеджер должен описать тип одной фразой. Имена присваиваются **правилом по профилю**, а не вручную по номеру: номера кластеров зависят от инициализации, а правило воспроизводимо при любом перезапуске.

Порядок присвоения:

1. класс с наименьшей активностью → **Спящий пластик**;
2. из оставшихся класс с наибольшим средним остатком → **Хранители остатка**;
3. из оставшихся класс с наибольшим оборотом → **Обороты**;
4. из оставшихся класс с наибольшей долей интернет-платежей → **Цифровые тратчики**;
5. последний → **Наличные повседневные**.

Перед расчётом денег проверяем, что именно лежит в `net_revenue_uzs`: входит ли туда абонентская плата. От этого зависит, окупается ли клиент.

In [ ]:
# --- Что именно считается доходом банка ---
chk_rev = q("""
    SELECT round(avg(net_revenue_uzs), 1)                                        AS факт,
           round(avg(interchange_uzs + float_income_uzs + transfer_fee_uzs
                     - atm_cost_uzs), 1)                                          AS сумма_компонент
    FROM public.seg_revenue_monthly
    WHERE month_start >= DATE '%s' AND month_start < DATE '%s'
""" % (W_START, W_END))
display(chk_rev)
FEE_INCLUDED = abs(float(chk_rev.факт[0]) - float(chk_rev.сумма_компонент[0])) > 1000
print('Абонентская плата %s в net_revenue_uzs.'
      % ('входит' if FEE_INCLUDED else 'НЕ входит'))
print('Значит при расчёте окупаемости её %s.'
      % ('учитывать отдельно не нужно' if FEE_INCLUDED else 'надо прибавить'))

# --- Присвоение имён по правилу ---
prof = F.assign(кластер=base)
med = prof.groupby('кластер').median(numeric_only=True)

names, free = {}, list(med.index)
def take(col, how='max'):
    sub = med.loc[free, col]
    return (sub.idxmax() if how == 'max' else sub.idxmin())

c_ = take('операций_в_месяц', 'min'); names[c_] = 'Спящий пластик';        free.remove(c_)
c_ = take('остаток_средний');        names[c_] = 'Хранители остатка';     free.remove(c_)
c_ = take('расход_сумма');           names[c_] = 'Обороты';               free.remove(c_)
c_ = take('доля_канал_ecom');        names[c_] = 'Цифровые тратчики';     free.remove(c_)
names[free[0]] = 'Наличные повседневные'

prof['тип'] = prof.кластер.map(names)
print()
print('Присвоенные имена:', {int(k): v for k, v in names.items()})

# --- Карточки типов ---
fee = float(q('SELECT monthly_fee_uzs FROM public.seg_tariff_current').iloc[0, 0])
card = prof.groupby('тип').agg(
    клиентов=('customer_id', 'size'),
    операций_в_месяц=('операций_в_месяц', 'median'),
    оборот_за_окно=('расход_сумма', 'median'),
    средний_чек=('средний_чек', 'median'),
    остаток=('остаток_средний', 'median'),
    доля_atm=('доля_канал_atm', 'median'),
    доля_ecom=('доля_канал_ecom', 'median'),
    повседневные=('доля_повседневные', 'median'),
    дискреционные=('доля_дискреционные', 'median'),
    стоимость_снятий=('стоимость_снятий', 'median'),
    доход_всего=('доход_банка', 'sum'),
)
card['доля_клиентов_%'] = (100 * card.клиентов / len(prof)).round(1)
card['доход_на_клиента_мес'] = (card.доход_всего / card.клиентов / W_MONTHS).round(0)
card['вклад_в_доход_%'] = (100 * card.доход_всего / card.доход_всего.sum()).round(1)
if not FEE_INCLUDED:
    card['с_абонплатой_мес'] = card.доход_на_клиента_мес + fee
card['окупает_абонплату'] = card.доход_на_клиента_мес > fee

order = ['Обороты', 'Хранители остатка', 'Цифровые тратчики',
         'Наличные повседневные', 'Спящий пластик']
order = [o for o in order if o in card.index]
display(card.loc[order, ['клиентов', 'доля_клиентов_%', 'доход_на_клиента_мес',
                         'вклад_в_доход_%', 'окупает_абонплату']].round(1))
print()
display(card.loc[order, ['операций_в_месяц', 'оборот_за_окно', 'средний_чек', 'остаток',
                         'доля_atm', 'доля_ecom', 'повседневные', 'дискреционные',
                         'стоимость_снятий']].round(2))

print()
print('Ключевая асимметрия для бизнеса:')
top = card.loc[order[0]]
big = card.sort_values('клиентов', ascending=False).iloc[0]
print('  %s: %.1f%% клиентов дают %.1f%% дохода'
      % (order[0], top['доля_клиентов_%'], top['вклад_в_доход_%']))
print('  %s: %.1f%% клиентов дают %.1f%% дохода'
      % (big.name, big['доля_клиентов_%'], big['вклад_в_доход_%']))


### 8.2 Что предлагаем каждому типу

Действующий тариф один на всех: 15 000 сум в месяц, 2 млн бесплатных снятий, 0,5% за перевод. Он одинаково невыгоден и тем, кто снимает наличные, и тем, кто держит остаток.

| Тип | Кто это одной фразой | Что предлагаем | Зачем банку |
|---|---|---|---|
| **Обороты** | Платит картой постоянно и на большие суммы, половина операций — в интернете и рознице | Премиальный тариф с повышенной абонплатой и кешбэком по дискреционным категориям | Дают около половины всего дохода при восьмой части базы — защита от ухода к конкуренту |
| **Хранители остатка** | Операций мало, но на счету держит крупный остаток | Тариф с бесплатным обслуживанием при неснижаемом остатке, накопительный счёт | Доход идёт от размещения средств, а не от операций; абонплата их раздражает без причины |
| **Цифровые тратчики** | Почти все платежи в интернете, траты необязательные | Кешбэк по онлайн-покупкам, рассрочка на маркетплейсах | Высокая доля интерчейнджа, дешёвый в обслуживании канал |
| **Наличные повседневные** | Снимает наличные в банкомате, тратит на продукты и коммуналку | Тариф с платными снятиями сверх сниженного лимита либо бонус за оплату картой | Снятия стоят банку денег; надо либо переводить в безнал, либо перекладывать стоимость |
| **Спящий пластик** | Карта оформлена, но почти не используется | Активационная кампания либо перевод на бесплатный тариф без обслуживания | Не окупает абонплату; держать его на платном тарифе — терять на обслуживании |

**Решение бизнеса, которое меняется** (это ответ на Q10). Сейчас продуктовая команда собирается делить линейку по обороту, потому что других данных нет. Типология показывает, что оборот не главная ось: **Хранители остатка** приносят банку сопоставимо с **Цифровыми тратчиками**, но по обороту попадают в разные квартили, а по природе дохода отличаются полностью — у одних это размещение средств, у других интерчейндж. Тариф, построенный по квартилям оборота, поставил бы их в один ряд и потерял бы обе группы.

Второе решение: **Наличные повседневные** — самый крупный тип базы, но по доходу он далеко не первый, и значительную часть дохода съедает стоимость снятий. Это либо кандидат на изменение лимита бесплатных снятий, либо на кампанию по переводу в безналичные платежи. Без типологии он растворился бы в середине распределения по обороту.

---

## Раздел 10. Экономический эффект тарифной реформы

Типология без денег не меняет решений. Но считать эффект надо в два слоя, и путать их нельзя.

**Слой первый — механический.** Что произойдёт при неизменном поведении клиентов. Считается прямо из данных: мы знаем помесячные снятия каждого клиента, значит знаем объём сверх любого лимита; знаем дискреционные траты, значит знаем стоимость кешбэка; знаем остатки, значит знаем, кто проходит порог. Допущений здесь нет.

**Слой второй — поведенческий.** Перейдут ли наличные в безнал, уйдут ли оборотистые при повышении абонплаты, проснутся ли спящие. **Из этих данных не выводится вообще**: в базе нет ни одного случая смены тарифа, эластичность измерить не на чем. Здесь только сценарии с явно названными числами.

Разделение принципиальное. Первый слой можно защищать как расчёт, второй — только как гипотезу, требующую пилота.

### Предлагаемая линейка

Четыре тарифа на пять типов. Пятый тариф не нужен: «Спящий пластик» — это кампания активации, а не тарифный план.

| Тариф | Для кого | Абонплата | Бесплатные снятия | Кешбэк |
|---|---|---|---|---|
| **Премиальный** | Обороты | 50 000 | 10 млн, далее 0,5% | 20% интерчейнджа с дискреционных |
| **Остаток** | Хранители остатка | 0 при остатке от 10 млн, иначе 15 000 | 2 млн, далее 0,5% | — |
| **Цифровой** | Цифровые тратчики | 15 000 | 500 тыс., далее 0,7% | 25% интерчейнджа с онлайн-покупок |
| **Базовый** | Наличные, Спящие | 15 000 | 500 тыс., далее 0,7% | 15% интерчейнджа за оплату картой |

**Кешбэк задан долей интерчейнджа, а не процентом от оборота.** Это не стилистика. Назначенная наугад ставка вроде «1% на онлайн» легко превышает то, что банк зарабатывает на этих же транзакциях, и программа лояльности становится убыточной по построению. Привязка к интерчейнджу делает её самоограничивающейся: мы возвращаем часть полученного, а фактическая ставка выводится из данных в расчёте ниже.

**Логика линейки: перераспределить выручку, а не раздать её.**

- Обороты платят 400 403 сум операционного дохода в месяц — абонплата 50 000 составляет 12% от него, банк не рискует. Кешбэк 0,5% ограничен так, чтобы финансироваться из интерчейнджа. У них же самая высокая стоимость снятий, поэтому щедрый лимит — реальная ценность, а не декорация.
- Хранители приносят деньги остатком, а не операциями: 17,1 млн под 10% годовых дают около 142 тысяч в месяц против абонплаты в 15 тысяч. Брать с них плату за обслуживание — раздражать источник дохода ради одной десятой этого дохода.
- Цифровые обходятся банку дешевле всех: стоимость снятий 1,3 тыс. в месяц против 19,7 у Оборотов. Дешёвый канал позволяет снизить плату и вернуть часть интерчейнджа кешбэком. Малый лимит снятий их не ущемляет — доля банкоматов у них 10%.
- Наличные создают затраты, которые сейчас никто не оплачивает: бесплатный лимит в 2 млн покрывает почти весь их объём снятий. Снижение лимита создаёт мотив платить картой, бонус даёт альтернативу штрафу.

**Чего мы НЕ делаем и почему.** Не обнуляем абонплату «Наличным повседневным», хотя это первое, что просится для массового сегмента. У них 35% выручки — абонплата; обнуление для 7 212 человек стоит около 108 млн сум в месяц, и никакой комиссией за снятия это не отбивается.

In [ ]:
# ---------- Параметры новой линейки. Все числа здесь, чтобы их можно было крутить ----------
# Кешбэк задаётся НЕ абсолютной ставкой, а долей интерчейнджа, который банк
# зарабатывает на этих же транзакциях. Так он не может стать убыточным по построению:
# возвращаем часть того, что получили, а не процент от оборота наугад.
#
# Второе ограничение, которое проверяется ниже: кешбэк не должен превышать 10%
# операционного дохода типа. Интерчейндж — не вся прибыль с клиента, и у типов
# с низкими остатками он составляет заметно большую её долю. Без этой проверки
# щедрость к одному типу оплачивается из кармана банка, а не из его же транзакций.
CB_LIMIT_OF_INCOME = 0.10
TARIFFS = {
    'Обороты':               dict(тариф='Премиальный', fee=50_000, atm_free=10_000_000,
                                  atm_rate=0.005, cb_share=0.20, cb_base='дискреционные'),
    'Хранители остатка':     dict(тариф='Остаток',     fee=0,      atm_free=2_000_000,
                                  atm_rate=0.005, cb_share=0.0,   cb_base=None,
                                  fee_if_below=15_000, balance_threshold=10_000_000),
    'Цифровые тратчики':     dict(тариф='Цифровой',    fee=15_000, atm_free=500_000,
                                  atm_rate=0.007, cb_share=0.22,  cb_base='ecom'),
    'Наличные повседневные': dict(тариф='Базовый',     fee=15_000, atm_free=500_000,
                                  atm_rate=0.007, cb_share=0.13,  cb_base='безнал_бонус'),
    'Спящий пластик':        dict(тариф='Базовый',     fee=15_000, atm_free=500_000,
                                  atm_rate=0.007, cb_share=0.13,  cb_base='безнал_бонус'),
}
CUR_FEE, CUR_FREE, MONTHS = 15_000, 2_000_000, W_MONTHS

# ---------- Помесячные факты по каждому клиенту ядра ----------
# Снятия: объём сверх лимита считается ПО МЕСЯЦАМ, а не за окно целиком —
# лимит месячный, и суммирование за полгода завысило бы облагаемую базу втрое.
atm_m = q("""
    SELECT customer_id, date_trunc('month', txn_ts)::date AS месяц,
           sum(amount_uzs) AS снято
    FROM %s
    WHERE txn_ts >= DATE '%s' AND txn_ts < DATE '%s'
      AND direction = 'out' AND channel = 'atm'
    GROUP BY 1, 2
""" % (TXN, W_START, W_END))

# Безналичные траты по типам категорий — база для кешбэка
spend_m = q("""
    SELECT customer_id,
           sum(amount_uzs) FILTER (WHERE channel = 'ecom')          AS онлайн,
           sum(amount_uzs) FILTER (WHERE channel IN ('pos','ecom')) AS безнал,
           sum(amount_uzs) FILTER (WHERE mcc_group IN
               ('restaurants','entertainment','travel','clothing','electronics','marketplace'))
                                                                     AS дискреционные
    FROM %s
    WHERE txn_ts >= DATE '%s' AND txn_ts < DATE '%s' AND direction = 'out'
    GROUP BY 1
""" % (TXN, W_START, W_END))

bal_m = q("""
    SELECT customer_id, avg(avg_balance_uzs) AS остаток_средний
    FROM public.seg_balances
    WHERE month_start >= DATE '%s' AND month_start < DATE '%s'
    GROUP BY 1
""" % (W_START, W_END))

E = prof[['customer_id', 'тип', 'доход_банка']].copy()
E = (E.merge(spend_m, on='customer_id', how='left')
       .merge(bal_m, on='customer_id', how='left')
       .fillna(0))

print('Клиентов в расчёте: %s' % format(len(E), ',d').replace(',', ' '))
print('Месяцев в окне: %d' % MONTHS)
display(E.groupby('тип').agg(клиентов=('customer_id', 'size'),
                             онлайн_за_окно=('онлайн', 'median'),
                             дискреционные_за_окно=('дискреционные', 'median')).round(0))


### 10.1 Выручка сегодня: сколько из неё абонплата

Прежде чем менять тариф, надо понять структуру текущей выручки. Абонентская плата в `net_revenue_uzs` не входит, поэтому полная выручка с клиента — это операционный доход плюс 15 000 в месяц.

Доля абонплаты показывает, насколько устойчив доход с каждого типа. Если она велика, банк живёт не на поведении клиента, а на плате за сам факт наличия карты.

In [ ]:
cur = E.groupby('тип').agg(клиентов=('customer_id', 'size'),
                           опер_доход_окно=('доход_банка', 'sum'))
cur['опер_доход_мес_на_клиента'] = (cur.опер_доход_окно / cur.клиентов / MONTHS).round(0)
cur['абонплата_мес_всего'] = cur.клиентов * CUR_FEE
cur['опер_доход_мес_всего'] = (cur.опер_доход_окно / MONTHS).round(0)
cur['выручка_мес_всего'] = cur.абонплата_мес_всего + cur.опер_доход_мес_всего
cur['доля_абонплаты_%'] = (100 * cur.абонплата_мес_всего / cur.выручка_мес_всего).round(1)
cur['выручка_мес_на_клиента'] = (cur.выручка_мес_всего / cur.клиентов).round(0)

order = ['Обороты', 'Хранители остатка', 'Цифровые тратчики',
         'Наличные повседневные', 'Спящий пластик']
order = [o for o in order if o in cur.index]
display(cur.loc[order, ['клиентов', 'опер_доход_мес_на_клиента', 'выручка_мес_на_клиента',
                        'выручка_мес_всего', 'доля_абонплаты_%']])

BASE_REVENUE = cur.выручка_мес_всего.sum()
print()
print('Полная выручка с ядра базы: %s сум в месяц'
      % format(int(BASE_REVENUE), ',d').replace(',', ' '))
print('Из них абонентская плата:   %s сум (%.1f%%)'
      % (format(int(cur.абонплата_мес_всего.sum()), ',d').replace(',', ' '),
         100 * cur.абонплата_мес_всего.sum() / BASE_REVENUE))
print()
print('Чем выше доля абонплаты в типе, тем менее устойчив доход с него:')
print('банк получает деньги не за поведение, а за сам факт наличия карты.')


### 10.2 Механический эффект: что будет при неизменном поведении

Здесь никаких допущений. Считаем по фактическим данным:

- **комиссия за снятия** — объём сверх нового лимита, посчитанный **по месяцам**, а не за окно целиком (лимит месячный, суммирование за полгода завысило бы базу втрое);
- **стоимость кешбэка** — процент от фактических трат в соответствующих категориях;
- **абонплата** — новая ставка, для тарифа «Остаток» с проверкой порога по каждому клиенту.

Результат — дельта к текущей выручке при условии, что клиенты ведут себя ровно так же.

In [ ]:
# --- Комиссия за снятия сверх лимита, помесячно ---
atm_e = atm_m.merge(E[['customer_id', 'тип']], on='customer_id', how='inner')
atm_e['лимит'] = atm_e.тип.map(lambda t: TARIFFS[t]['atm_free'])
atm_e['ставка'] = atm_e.тип.map(lambda t: TARIFFS[t]['atm_rate'])
atm_e['сверх_лимита'] = (atm_e.снято - atm_e.лимит).clip(lower=0)
atm_e['комиссия'] = atm_e.сверх_лимита * atm_e.ставка

# то же при действующем лимите — чтобы считать именно ДЕЛЬТУ
atm_e['сверх_старого'] = (atm_e.снято - CUR_FREE).clip(lower=0)
atm_e['комиссия_старая'] = 0.0        # сейчас комиссии за снятия нет

atm_by_type = atm_e.groupby('тип').agg(
    комиссия_за_окно=('комиссия', 'sum'),
    объём_сверх_лимита=('сверх_лимита', 'sum'),
    месяцев_с_превышением=('сверх_лимита', lambda s: int((s > 0).sum())))

# --- Ставка интерчейнджа, выведенная из данных ---
_ic = q("""
    SELECT sum(interchange_uzs) AS ic FROM public.seg_revenue_monthly
    WHERE month_start >= DATE '%s' AND month_start < DATE '%s'
""" % (W_START, W_END)).ic[0]
_noncash = q("""
    SELECT sum(amount_uzs) AS s FROM %s
    WHERE txn_ts >= DATE '%s' AND txn_ts < DATE '%s'
      AND direction = 'out' AND channel IN ('pos','ecom')
""" % (TXN, W_START, W_END)).s[0]
IC_RATE = float(_ic) / float(_noncash)
print('Интерчейндж по факту: %.3f%% от безналичного оборота' % (100 * IC_RATE))
print('Ставки кешбэка, выведенные из него:')
for t_, cfg in TARIFFS.items():
    if cfg['cb_base']:
        print('  %-24s %.3f%% (%.0f%% интерчейнджа)'
              % (t_, 100 * cfg['cb_share'] * IC_RATE, 100 * cfg['cb_share']))
print()

# --- Стоимость кешбэка ---
def cashback(row):
    t = TARIFFS[row.тип]
    rate = t['cb_share'] * IC_RATE
    if t['cb_base'] == 'дискреционные':
        return row.дискреционные * rate
    if t['cb_base'] == 'ecom':
        return row.онлайн * rate
    if t['cb_base'] == 'безнал_бонус':
        return row.безнал * rate
    return 0.0

E['кешбэк_за_окно'] = E.apply(cashback, axis=1)

# --- Новая абонплата, с проверкой порога для тарифа "Остаток" ---
def new_fee(row):
    t = TARIFFS[row.тип]
    if 'balance_threshold' in t:
        return t['fee'] if row.остаток_средний >= t['balance_threshold'] else t['fee_if_below']
    return t['fee']

E['абонплата_новая_мес'] = E.apply(new_fee, axis=1)

# --- Сводка ---
mech = E.groupby('тип').agg(
    клиентов=('customer_id', 'size'),
    кешбэк_окно=('кешбэк_за_окно', 'sum'),
    абонплата_новая_мес=('абонплата_новая_мес', 'sum'))
mech = mech.join(atm_by_type[['комиссия_за_окно']]).fillna(0)

mech['абонплата_старая_мес'] = mech.клиентов * CUR_FEE
mech['комиссия_мес'] = (mech.комиссия_за_окно / MONTHS).round(0)
mech['кешбэк_мес'] = (mech.кешбэк_окно / MONTHS).round(0)
mech['дельта_абонплата'] = mech.абонплата_новая_мес - mech.абонплата_старая_мес
mech['дельта_итого_мес'] = (mech.дельта_абонплата + mech.комиссия_мес - mech.кешбэк_мес).round(0)
mech['дельта_на_клиента'] = (mech.дельта_итого_мес / mech.клиентов).round(0)

display(mech.loc[order, ['клиентов', 'дельта_абонплата', 'комиссия_мес', 'кешбэк_мес',
                         'дельта_итого_мес', 'дельта_на_клиента']])

def sgn(x):
    return ('+' if x >= 0 else '-') + format(int(abs(x)), ',d').replace(',', ' ')

MECH_TOTAL = mech.дельта_итого_мес.sum()
print()
print('Механический эффект: %s сум в месяц (%+.1f%% к текущей выручке)'
      % (sgn(MECH_TOTAL), 100 * MECH_TOTAL / BASE_REVENUE))
print('За год: %s сум' % sgn(MECH_TOTAL * 12))
print()
print('Проверка: не превышает ли кешбэк 10%% операционного дохода типа')
_chk = mech.assign(кешбэк_к_доходу_pct=(100 * mech.кешбэк_мес * MONTHS
                                        / cur.опер_доход_окно).round(1))
_chk['в_пределах'] = _chk.кешбэк_к_доходу_pct <= 100 * CB_LIMIT_OF_INCOME
display(_chk.loc[order, ['кешбэк_мес', 'кешбэк_к_доходу_pct', 'в_пределах']])
if not _chk.в_пределах.all():
    print()
    print('[!] Для типов выше кешбэк съедает больше десятой части дохода.')
    print('    Снизьте cb_share в конфиге: интерчейндж — не вся прибыль с клиента,')
    print('    и у типов с низкими остатками он составляет большую её долю.')
print()
print('Сколько клиентов проходит порог остатка 10 млн в тарифе «Остаток»:')
h = E[E.тип == 'Хранители остатка']
print('  %d из %d (%.1f%%) — платят 0, остальные 15 000'
      % (int((h.остаток_средний >= 10_000_000).sum()), len(h),
         100 * (h.остаток_средний >= 10_000_000).mean()))


### 10.3 Поведенческий слой: три сценария

Всё, что выше, посчитано при условии «клиенты ведут себя как раньше». В жизни так не бывает: тариф меняет поведение — в этом и смысл реформы.

**Проблема в том, что измерить это на наших данных нельзя.** В базе единственный тариф, ни одного случая перехода, эластичность спроса оценить не на чем. Поэтому дальше — не расчёт, а сценарии с явно названными числами.

Три параметра, каждый из которых требует подтверждения продуктовым блоком:

1. **Переток наличных в безнал** — какая доля снятий у «Наличных повседневных» превратится в оплату картой. Влияет на интерчейндж (плюс) и на комиссию за снятия (минус).
2. **Отток Оборотов при росте абонплаты втрое** — какая доля уйдёт к конкуренту. Самый дорогой риск: каждый ушедший уносит 400 тысяч операционного дохода в месяц.
3. **Активация Спящих** — какая доля начнёт пользоваться картой после кампании.

Три набора значений: пессимистичный, базовый, оптимистичный.

In [ ]:
SCENARIOS = {
    'пессимистичный': dict(переток_в_безнал=0.05, отток_оборотов=0.10, активация_спящих=0.05),
    'базовый':        dict(переток_в_безнал=0.15, отток_оборотов=0.04, активация_спящих=0.15),
    'оптимистичный':  dict(переток_в_безнал=0.30, отток_оборотов=0.01, активация_спящих=0.30),
}

atm_cash_m = atm_e.groupby('тип').снято.sum() / MONTHS      # объём снятий в месяц по типам
oborot_rev_mes = cur.loc['Обороты', 'выручка_мес_на_клиента']
n_oborot = int(cur.loc['Обороты', 'клиентов'])
n_sleep = int(cur.loc['Спящий пластик', 'клиентов'])
sleep_target_rev = cur.loc['Наличные повседневные', 'опер_доход_мес_на_клиента']

rows = []
for name, sc in SCENARIOS.items():
    # 1. Переток наличных в безнал
    moved = atm_cash_m.get('Наличные повседневные', 0) * sc['переток_в_безнал']
    plus_ic = moved * IC_RATE
    lost_atm_fee = moved * TARIFFS['Наличные повседневные']['atm_rate']
    extra_cb = moved * TARIFFS['Наличные повседневные']['cb_share'] * IC_RATE

    # 2. Отток Оборотов
    lost_ob = n_oborot * sc['отток_оборотов'] * oborot_rev_mes

    # 3. Активация Спящих: доходят до уровня «Наличных повседневных»
    gained_sleep = (n_sleep * sc['активация_спящих']
                    * (sleep_target_rev - cur.loc['Спящий пластик', 'опер_доход_мес_на_клиента']))

    beh = plus_ic - lost_atm_fee - extra_cb - lost_ob + gained_sleep
    rows.append({
        'сценарий': name,
        'переток_в_безнал': '%.0f%%' % (100 * sc['переток_в_безнал']),
        'отток_оборотов': '%.0f%%' % (100 * sc['отток_оборотов']),
        'активация_спящих': '%.0f%%' % (100 * sc['активация_спящих']),
        'интерчейндж_плюс': round(plus_ic),
        'потеря_комиссии_atm': round(-lost_atm_fee),
        'кешбэк_доп': round(-extra_cb),
        'потеря_оборотов': round(-lost_ob),
        'активация_плюс': round(gained_sleep),
        'поведенческий_итог': round(beh),
        'ВСЕГО_мес': round(MECH_TOTAL + beh),
    })

S = pd.DataFrame(rows).set_index('сценарий')
display(S[['переток_в_безнал', 'отток_оборотов', 'активация_спящих',
           'поведенческий_итог', 'ВСЕГО_мес']])
print()
display(S[['интерчейндж_плюс', 'потеря_комиссии_atm', 'кешбэк_доп',
           'потеря_оборотов', 'активация_плюс']])

print()
print('Механический эффект (без допущений): %s сум/мес' % sgn(MECH_TOTAL))
for n in S.index:
    print('  %-16s итого %s сум/мес  (%+.1f%% к выручке, за год %s)'
          % (n, sgn(S.loc[n, 'ВСЕГО_мес']),
             100 * S.loc[n, 'ВСЕГО_мес'] / BASE_REVENUE,
             sgn(S.loc[n, 'ВСЕГО_мес'] * 12)))


### 10.4 Чувствительность: какой параметр решает исход

Три допущения не равнозначны. Если результат целиком определяется одним из них, то именно его надо измерять пилотом в первую очередь, а остальные можно взять грубо.

Меняем каждый параметр по отдельности, остальные держим на базовом уровне.

In [ ]:
base_sc = SCENARIOS['базовый']

def effect(переток, отток, активация):
    moved = atm_cash_m.get('Наличные повседневные', 0) * переток
    beh = (moved * IC_RATE
           - moved * TARIFFS['Наличные повседневные']['atm_rate']
           - moved * TARIFFS['Наличные повседневные']['cb_share'] * IC_RATE
           - n_oborot * отток * oborot_rev_mes
           + n_sleep * активация * (sleep_target_rev
                                    - cur.loc['Спящий пластик', 'опер_доход_мес_на_клиента']))
    return MECH_TOTAL + beh

GRID = {
    'переток в безнал':  [0.0, 0.05, 0.15, 0.30, 0.50],
    'отток Оборотов':    [0.0, 0.02, 0.04, 0.10, 0.20],
    'активация Спящих':  [0.0, 0.05, 0.15, 0.30, 0.50],
}

def with_param(name, v):
    p = dict(base_sc)
    key = {'переток в безнал': 'переток_в_безнал',
           'отток Оборотов': 'отток_оборотов',
           'активация Спящих': 'активация_спящих'}[name]
    p[key] = v
    return effect(p['переток_в_безнал'], p['отток_оборотов'], p['активация_спящих'])

# Отдельная таблица на каждый параметр: сетки значений разные, общий индекс их путает
for name, grid in GRID.items():
    t = pd.DataFrame({
        'значение': ['%.0f%%' % (100 * v) for v in grid],
        'итого, сум/мес': [round(with_param(name, v)) for v in grid],
    })
    t['к выручке, %'] = (100 * t['итого, сум/мес'] / BASE_REVENUE).round(2)
    print(name.upper())
    display(t.set_index('значение'))

rng_ = pd.DataFrame({
    'min': {n: min(with_param(n, v) for v in g) for n, g in GRID.items()},
    'max': {n: max(with_param(n, v) for v in g) for n, g in GRID.items()},
})
rng_['размах'] = (rng_['max'] - rng_['min']).round(0)
print()
print('Размах результата при изменении каждого параметра:')
display(rng_.sort_values('размах', ascending=False).round(0))

crit = rng_.размах.idxmax()
weak = rng_.размах.idxmin()
print()
print('Решающий параметр: %s (размах %s сум/мес).'
      % (crit, format(int(rng_.loc[crit, 'размах']), ',d').replace(',', ' ')))
print('Наименее важный: %s (размах %s сум/мес) — на итог почти не влияет.'
      % (weak, format(int(rng_.loc[weak, 'размах']), ',d').replace(',', ' ')))
print('Именно его надо измерять пилотом в первую очередь. Остальные допущения')
print('влияют на итог слабее и на этапе решения могут быть взяты грубо.')

# Точка безубыточности по решающему параметру
if crit == 'отток Оборотов':
    for от in np.arange(0, 0.51, 0.005):
        if effect(base_sc['переток_в_безнал'], от, base_sc['активация_спящих']) < 0:
            print()
            print('Реформа уходит в минус при оттоке Оборотов выше %.1f%% '
                  '(это %d человек из %d).' % (100 * от, int(n_oborot * от), n_oborot))
            break


### 10.5 Итог и что должен подтвердить бизнес

**Что можно защищать как расчёт.** Механический эффект — он посчитан из фактических помесячных данных без единого допущения о поведении. Это нижняя граница при условии, что никто не поменяет привычек и никто не уйдёт.

**Что остаётся гипотезой.** Всё остальное. В базе один тариф и ни одного случая перехода, поэтому эластичность измерить не на чем. Диапазон между пессимистичным и оптимистичным сценарием — это честная мера нашего незнания, а не прогноз.

**Что должен подтвердить продуктовый и финансовый блок перед запуском:**

1. **Готовность Оборотов платить втрое больше.** Это решающий параметр — при оттоке выше точки безубыточности вся реформа уходит в минус. Проверяется опросом и пилотом на подвыборке, а не расчётом.
2. **Ставку размещения средств.** Тариф «Остаток» обнуляет абонплату в расчёте на доход от размещения. Мы взяли ориентир, реальную ставку фондирования знает казначейство.
3. **Стоимость кешбэка как маркетингового расхода.** Мы считаем его прямым вычетом из выручки. Если он финансируется из маркетингового бюджета, картина меняется.
4. **Регуляторные ограничения** на комиссию за снятие наличных и на минимальный остаток.

**Чего мы сознательно не считали.** Стоимость самой миграции: разработка, коммуникация, поддержка, отток из-за раздражения сменой условий. Это отдельная смета, и без неё приведённые цифры — валовый эффект, а не чистый.

**Как это соотносится с типологией.** Заметьте, что вся экономика построена на различиях, которые видит только типология: лимит снятий имеет смысл менять для «Наличных», а не для всех; порог остатка работает для «Хранителей», а не для верхнего квартиля по обороту. Разбиение по квартилям оборота не дало бы ни одного из этих параметров.

---

## Раздел 9. Ответы на вопросы формы

**Q1. Сколько всего клиентов в `seg_customers`** — 20 000.

**Q2. Сколько клиентов не совершили ни одной операции за весь период** — 245.

**Q3. У скольких история операций короче 6 месяцев** — 782.

**Q4. Сколько клиентов, у которых больше половины исходящих операций — снятия в банкомате** — 701.

*Все четыре считаются по `seg_transactions`. Архив не подключается: он полный дубликат (все 2 718 367 строк нашлись в основной по `txn_id`), плюс внутри него 79 199 задвоений.*

---

**Q5. За какой период считали признаки и почему. Что обнаружили, посмотрев по месяцам.**

Признаки считаны за последние шесть месяцев — с 1 июля по 28 декабря 2025. При взгляде по месяцам обнаружилось, что число клиентов в помесячных таблицах растёт монотонно с 15 013 в январе 2024 до 20 000 в октябре 2025: это не активность, а накопление регистраций, и признаки за все 24 месяца дали бы типологию по дате прихода, а не по поведению. Шесть месяцев — компромисс: окно покрывает 97,4% базы (при годовом было бы 90%), но остаётся достаточно длинным, чтобы доли по категориям трат не были шумом. Дополнительно проверили стабильность: медиана изменения активности между полугодиями 1,03, то есть типичный клиент ведёт себя одинаково, но у 7,7% поведение меняется резко — активность более чем вдвое либо доля снятий сдвигается больше чем на 30 пунктов.

---

**Q6. По каким осям делили. Какие таблицы не использовали. Что думаете про готовый сегмент.**

Три оси: масштаб и интенсивность операций, структура поведения в долях (каналы и укрупнённые категории трат), деньги для банка (остатки, стоимость снятий, продукты). Доли выбраны вместо сумм сознательно — они не зависят от масштаба и поэтому не дублируют первую ось. Шестнадцать категорий свёрнуты в три группы, иначе структура получила бы двадцать голосов из тридцати пяти и определила бы всё разбиение. Не использованы: стаж и дата регистрации (вернули бы типологию по времени прихода), пол и регион (анкета, а не поведение, плюс регуляторный риск для тарифа), `seg_transactions_archive` (дубликат), `seg_temp` (пустая), `seg_rfm_old` (чужой расчёт за неизвестный период — воспроизвести и защитить нельзя), `seg_customers_360` (`txn_cnt_total` и `turnover_total` посчитаны за всю историю, то есть ровно та ловушка с накоплением, которую мы обходим).

Про `crm_segment` из старой витрины: **это случайная метка, а не сегментация.** Группы A, B, C, D распределены почти поровну (5 076, 4 870, 5 074, 4 980), но главное — они неразличимы по поведению:

| Сегмент | Клиентов | Медиана операций | Медиана оборота |
|---|---|---|---|
| A | 5 076 | 342 | 128,7 млн |
| B | 4 870 | 338 | 127,5 млн |
| C | 5 074 | 347 | 125,5 млн |
| D | 4 980 | 348 | 132,6 млн |

Тест Краскела-Уоллиса по обороту: H = 0,40, p = 0,941. Гипотеза о равенстве групп не отвергается даже близко — различий нет вовсе. Медианы совпадают с точностью до 3%, что для четырёх групп по пять тысяч человек означает случайное распределение.

Использовать такой сегмент как результат нельзя: он не несёт информации о клиенте. Выдать его за свою типологию — тем более.

Отдельно про агрегаты `seg_customers_360`: они посчитаны честно, корреляция `txn_cnt_total` и `turnover_total` с фактом по операциям равна 1,0. Но посчитаны они **за всю историю**, а длина истории у клиентов разная — именно поэтому мы их не берём.

---

**Q7. Сколько классов и как защищаете. Таблица устойчивости.**

Пять. Защита не сводится к локтю: силуэт достигает максимума ровно при k = 5 (0,306 против 0,284 при четырёх и 0,247 при шести), индекс Дэвиса-Болдина при k = 5 минимален (1,236), а падение инерции ломается после пятёрки — до неё каждый следующий класс забирает 14–15% остаточной дисперсии, после 7,5% и меньше. Содержательно: при k = 5 минимальный класс занимает 11,1% базы, при k = 7 появляется класс на 4,3% (753 человека), под который тариф строить нельзя; при k = 3 в один класс попадают клиенты с оборотом 43 и 232 млн, приносящие банку 377 тысяч и 1,76 миллиона.

| Проверка | ARI |
|---|---|
| `random_state` = 7 | 1,000 |
| `random_state` = 2024 | 1,000 |
| `random_state` = 999 | 1,000 |
| Половина выборки, попытка 1 | 0,994 |
| Половина выборки, попытка 2 | 0,994 |
| Половина выборки, попытка 3 | 0,993 |
| Без декабря (окно 5 месяцев) | 0,968 |
| Без 2 признаков, попытка 1 | 0,971 |
| Без 2 признаков, попытка 2 | 0,961 |
| Без 2 признаков, попытка 3 | 0,862 |

Средний ARI 0,974, минимальный 0,862. Самая чувствительная проверка — удаление признаков, что естественно: убирая пятую часть осей, мы меняем само пространство, в котором меряется близость. Устойчивость к инициализации полная.

*Оговорка, которую стоит назвать самому: значения такого уровня на реальных данных были бы подозрительны. Здесь они объясняются тем, что кластеры действительно хорошо разделены — силуэт имеет чёткий максимум, а данные синтетические и структура в них выражена сильнее, чем бывает в жизни.*

---

**Q8. Чем типология лучше квартилей оборота.**

Типология впереди по обеим осям. Доля объяснённой дисперсии (деньги считаются на логарифмической шкале — на сырых суммах показатель занижается для любого разбиения, потому что дисперсию создаёт десяток клиентов):

| | Типология (k=5) | Квартили оборота |
|---|---|---|
| Деньги, в среднем | **0,720** | 0,576 |
| Поведение, в среднем | **0,552** | 0,231 |
| Средний остаток | **0,608** | 0,266 |
| Доля снятий в банкомате | **0,455** | 0,101 |
| Доля платежей в интернете | **0,533** | 0,161 |
| Доля дискреционных трат | **0,856** | 0,364 |
| Оборот | 0,750 | 0,732 |

По самому обороту — паритет, и это честно: квартили строятся по нему. Разрыв возникает там, где строится тариф, — в структуре поведения, в два-четыре раза. ARI между двумя разбиениями 0,216, то есть типология не переоткрывает оборот дорогим способом.

Содержательный пример: **Хранители остатка** и **Цифровые тратчики** приносят банку 183 и 71 тысячу на клиента в месяц — деньги сопоставимые, но природа разная: размещение средств против интерчейнджа. По квартилям оборота они попали бы в разные группы и смешались бы там с непохожими клиентами.

---

**Q9. Как отнесёте нового клиента.**

Правило из двух шагов. Сначала когорта: нет операций в окне — «спящий», меньше полугода истории — «новичок». Затем тип: считаем те же четырнадцать признаков за последние шесть месяцев, применяем **сохранённые** параметры обработки — логарифм, границы обрезки по 1 и 99 процентилю, среднее и разброс для стандартизации — и находим ближайший центр кластера. Пересчитывать параметры на новых данных нельзя, иначе тип клиента зависел бы от того, кто ещё пришёл в тот же день.

Проверка: 3 508 клиентов (20%) отложены, кластеризация обучена на остальных 80%, отложенные размечены по правилу. **ARI между правилом и полной моделью — 0,999.** Пограничных случаев, где расстояние до двух ближайших центров различается меньше чем на 20%, — 8,1%. Для продуктовой команды правило упрощено до дерева глубины три: оно воспроизводит кластеризацию на 94,7% и читается глазами — первое разделение по доле дискреционных трат, второе по интенсивности операций.

---

**Q10. Перечислите типы: название, размер, поведение, доход, предложение.**

| Тип | Клиентов | Поведение | Доход на клиента в месяц | Вклад в доход | Предложение |
|---|---|---|---|---|---|
| **Обороты** | 2 196 (12,5%) | 81 операция в месяц, оборот 232 млн за полугодие, чек 650 тыс. | 400 403 сум | 49,2% | Премиальный тариф с повышенной абонплатой и кешбэком по дискреционным категориям |
| **Хранители остатка** | 2 395 (13,7%) | 9 операций в месяц, остаток 17,1 млн | 182 613 сум | 24,5% | Бесплатное обслуживание при неснижаемом остатке, накопительный счёт |
| **Цифровые тратчики** | 3 788 (21,6%) | 48 операций, 47% платежей в интернете, 49% трат дискреционные | 70 554 сум | 14,9% | Кешбэк по онлайн-покупкам, рассрочка на маркетплейсах |
| **Наличные повседневные** | 7 212 (41,1%) | 14 операций, 38% из них — снятия, 46% трат повседневные | 27 218 сум | 11,0% | Сниженный лимит бесплатных снятий либо бонус за оплату картой |
| **Спящий пластик** | 1 946 (11,1%) | 1,7 операции в месяц, дискреционных трат нет | 4 081 сум | 0,4% | Активационная кампания либо бесплатный тариф без обслуживания |

Плюс две когорты вне кластеризации: **новички** — 2 104 клиента, 101 350 сум в месяц, и **спящие** — 359 клиентов, 2 560 сум в месяц.

Важное уточнение по «Спящему пластику»: абонентская плата в `net_revenue_uzs` не входит. При операционном доходе 4 081 сум и абонплате 15 000 получается, что **четыре пятых того, что банк с него получает, — это плата за карту, которой не пользуются.** Такой доход не устойчив: он держится ровно до того дня, когда клиент заметит списание.

Плюс две когорты вне кластеризации: **новички** (2 104 клиента, меньше полугода истории) и **спящие** (359 клиентов, ни одной операции в окне).

**Какое решение меняется.** Продуктовая команда собиралась делить линейку по обороту. Типология показывает, что оборот — не главная ось: Хранители остатка и Цифровые тратчики приносят сопоставимые деньги разной природы и по квартилям оборота попали бы в разные группы, а внутри каждой группы смешались бы с непохожими клиентами. Второе: Наличные повседневные — самый крупный тип, но значительную часть дохода с него съедает стоимость снятий, и это прямой кандидат на пересмотр лимита бесплатных снятий.

---

**Дополнительно: экономический эффект тарифной реформы** (раздел 10)

Предлагаемая линейка — четыре тарифа на пять типов. Кешбэк везде задан долей интерчейнджа, который банк зарабатывает на тех же транзакциях, и ограничен десятой частью операционного дохода типа: иначе программа лояльности убыточна по построению. Фактическая ставка интерчейнджа, выведенная из данных, — 0,800% безналичного оборота.

| Тариф | Для кого | Абонплата | Бесплатные снятия | Кешбэк |
|---|---|---|---|---|
| Премиальный | Обороты | 50 000 | 10 млн, далее 0,5% | 0,160% на дискреционные |
| Остаток | Хранители | 0 при остатке от 10 млн | 2 млн, далее 0,5% | — |
| Цифровой | Цифровые тратчики | 15 000 | 500 тыс., далее 0,7% | 0,176% на онлайн |
| Базовый | Наличные, Спящие | 15 000 | 500 тыс., далее 0,7% | 0,104% за оплату картой |

**Механический эффект — при неизменном поведении, без единого допущения:**

| Тип | Абонплата | Комиссия за снятия | Кешбэк | Итого, сум/мес | На клиента |
|---|---|---|---|---|---|
| Обороты | +76,9 млн | +77,9 млн | −53,6 млн | **+101,5 млн** | +46 077 |
| Хранители остатка | −27,6 млн | +9,4 млн | 0 | −18,3 млн | −7 631 |
| Цифровые тратчики | 0 | +14,6 млн | −25,9 млн | −11,3 млн | −2 993 |
| Наличные повседневные | 0 | +49,4 млн | −18,0 млн | **+31,4 млн** | +4 354 |
| Спящий пластик | 0 | +0,1 млн | −0,4 млн | −0,3 млн | −158 |
| **Всего** | | | | **+102,7 млн** | |

Это **+5,0% к текущей выручке ядра базы (2 051 млн сум в месяц), или 1,23 млрд за год.**

**Структура выручки сегодня** объясняет, почему реформа вообще нужна: доля абонентской платы в выручке по типам — 3,6% у Оборотов, 7,6% у Хранителей, 17,5% у Цифровых, 35,5% у Наличных и **78,6% у Спящего пластика**. Чем выше доля, тем менее устойчив доход: банк получает деньги не за поведение, а за сам факт наличия карты.

**Что остаётся гипотезой.** Всё поведенческое. В базе один тариф и ни одного случая перехода, эластичность измерить не на чем. Анализ чувствительности показал два вывода:

- **Решает отток Оборотов.** Размах результата при изменении этого параметра — порядка 180 млн сум в месяц. Реформа уходит в минус при оттоке около 12% типа, то есть примерно 260 человек из 2 196. Пилот надо ставить именно на них.
- **Переток наличных в безнал экономически нейтрален.** Размах около 1 млн сум — потеря комиссии за снятия почти в точности компенсирует прирост интерчейнджа. Кампания по переводу в безнал имеет смысл для лояльности и снижения операционных затрат, но не для выручки.

**Риск, который надо назвать самому.** У «Наличных повседневных» тариф дорожает на 4 354 сум в месяц — это самый крупный тип базы, 41% клиентов. Обоснование есть: они создают затраты на снятия, которые сейчас не оплачивает никто, а бонус за оплату картой даёт им выход, которого раньше не было. Но это решение с политическими последствиями, и принимать его должен продуктовый блок, а не аналитик.

**Чего не считали:** стоимость самой миграции — разработку, коммуникацию, поддержку и отток из-за раздражения сменой условий. Приведённые цифры — валовый эффект, а не чистый.